In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os
import warnings
import gc

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings('ignore')

# ================== Device & Reproducibility ==================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Dataset (Spatial Domain Only) ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360 (spatial RGB images only)"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append(
                            (os.path.join(class_dir, img_name),
                             self.class_to_idx[class_name])
                        )
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # fallback black image (100x100 as in your dataset)
            return torch.zeros(3, 100, 100), label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with spatial-domain augmentation"""
    
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2,
                               saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0,
                                translate=(0.1, 0.1),
                                scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir  = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset  = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== Model: Original EfficientNet-B0 (3-channel input) ==================

class EfficientNetSpatialCNN(nn.Module):
    """
    EfficientNet-B0 based model operating purely in spatial domain.
    Uses original 3-channel input conv and standard EfficientNet feature extractor.
    Classifier is customized as in your original code.
    """
    def __init__(self, num_classes, dropout_rate=0.5):
        super(EfficientNetSpatialCNN, self).__init__()
        
        # Load pretrained EfficientNet-B0
        self.efficientnet = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
        
        # Number of features of B0
        num_features_last = 1280
        
        # Replace classifier with custom head
        self.efficientnet.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features_last, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        self._initialize_new_weights()
    
    def _initialize_new_weights(self):
        """Initialize new classifier layers with proper weights"""
        for m in self.efficientnet.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm1d,)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Just run EfficientNet normally (features + avgpool + classifier)
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        return self.efficientnet(x)

# ================== Training Utilities ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            self.counter = 0

def train_model(model, train_loader, val_loader,
                epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the EfficientNet spatial CNN model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        # Classifier is new
        if 'efficientnet.classifier' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    optimizer = torch.optim.AdamW(
        [
            {'params': pretrained_params, 'lr': lr * 0.01},
            {'params': new_params,       'lr': lr * 0.5}
        ],
        weight_decay=weight_decay
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses   = []
    train_accs   = []
    val_accs     = []
    
    best_val_acc = 0.0
    best_state   = None
    
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    
    for epoch in range(epochs):
        # ---------- Training ----------
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]",
                          leave=False)
        for i, (images, labels) in enumerate(train_pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            if torch.isnan(images).any() or torch.isinf(images).any():
                print(f"Warning: NaN/Inf detected in input batch {i}, skipping...")
                continue
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss in batch {i}, skipping...")
                    continue
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss in batch {i}, skipping...")
                    continue
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc':  f'{100 * correct_train / max(total_train, 1):.2f}%'
            })
            
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = running_loss / max(len(train_loader), 1)
        train_accuracy = 100 * correct_train / max(total_train, 1)
        train_losses.append(avg_train_loss)
        train_accs.append(train_accuracy)
        
        # ---------- Validation ----------
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total   = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]",
                            leave=False)
            for images, labels in val_pbar:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                if scaler is not None:
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss    = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss    = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total   += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc':  f'{100 * correct / max(total, 1):.2f}%'
                })
        
        avg_val_loss = running_val_loss / max(len(val_loader), 1)
        val_accuracy = 100 * correct / max(total, 1)
        val_losses.append(avg_val_loss)
        val_accs.append(val_accuracy)
        
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val   Loss: {avg_val_loss:.4f}, Val   Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_acc:.2f}%")
    
    return train_losses, val_losses, train_accs, val_accs

# ================== Plot Training Curves ==================

def plot_training_curves(train_losses, val_losses, train_accs, val_accs):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses,   'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss',  fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accs, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accs,   'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch',    fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Overall Metrics (Accuracy, Kappa, Precision, etc.) ==================

def compute_overall_metrics(all_labels, all_predictions, num_classes):
    """
    Compute overall accuracy, Cohen's kappa, macro precision/recall/F1,
    macro specificity, and error rate.
    """
    all_labels      = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    
    # Overall Accuracy
    acc = accuracy_score(all_labels, all_predictions)
    
    # Cohen's Kappa
    kappa = cohen_kappa_score(all_labels, all_predictions)
    
    # Macro Precision, Recall, F1
    precision = precision_score(all_labels, all_predictions,
                                average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_predictions,
                             average='macro', zero_division=0)
    f1        = f1_score(all_labels, all_predictions,
                         average='macro', zero_division=0)
    
    # Confusion matrix for specificity
    cm = confusion_matrix(all_labels, all_predictions, labels=range(num_classes))
    # Specificity per class: TN / (TN + FP)
    specificity_per_class = []
    for c in range(num_classes):
        TP = cm[c, c]
        FP = cm[:, c].sum() - TP
        FN = cm[c, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        denom = TN + FP
        if denom == 0:
            specificity_per_class.append(0.0)
        else:
            specificity_per_class.append(TN / denom)
    specificity_macro = float(np.mean(specificity_per_class))
    
    # Error rate
    error_rate = 1.0 * (1.0 - acc)
    
    metrics = {
        "accuracy": acc,
        "kappa": kappa,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "specificity_macro": specificity_macro,
        "error_rate": error_rate
    }
    return metrics

# ================== Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Spatial-Domain EfficientNet-B0 (Original Architecture)")
    print("Training on Fruits-360 (RGB only, no FFT, no dual-domain fusion)")
    print("="*80)
    
    # Dataset path - MODIFY THIS PATH
    data_root = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'
    
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please modify the 'data_root' variable to point to your dataset location.")
        return
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        print(f"Number of classes: {len(classes)}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size   = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples:   {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples:       {len(testset)}")
    
    # DataLoaders
    batch_size  = 128
    num_workers = 4 if os.name != 'nt' else 0
    
    print(f"\nBatch size:  {batch_size}")
    print(f"Num workers: {num_workers}")
    
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        testset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )
    
    print("\n[Step 3] Initializing EfficientNet-B0 (spatial only)...")
    model = EfficientNetSpatialCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("Architecture: EfficientNet-B0 backbone with custom classifier (spatial domain)")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"GPU memory reserved : {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
    print("\n[Step 4] Training model...")
    try:
        train_losses, val_losses, train_accs, val_accs = train_model(
            model, train_loader, val_loader,
            epochs=50,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[Step 4.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accs, val_accs)
    
    # ================== Evaluation on Test Set ==================
    
    print("\n[Step 5] Evaluating on test set (computing global metrics)...")
    model.eval()
    
    all_predictions = []
    all_labels      = []
    correct = 0
    total   = 0
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for images, labels in test_pbar:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            test_pbar.set_postfix({'acc': f'{100 * correct / max(total, 1):.2f}%'})
    
    test_accuracy = 100.0 * correct / max(total, 1)
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    # Compute additional metrics
    metrics = compute_overall_metrics(all_labels, all_predictions, num_classes=len(classes))
    
    print("\n[Step 6] Overall Test Metrics:")
    print("-" * 60)
    print(f"Overall Accuracy        : {metrics['accuracy'] * 100:.2f}%")
    print(f"Cohen's Kappa           : {metrics['kappa']:.4f}")
    print(f"Precision (Macro)       : {metrics['precision_macro'] * 100:.2f}%")
    print(f"Recall (Macro)          : {metrics['recall_macro'] * 100:.2f}%")
    print(f"F1 Score (Macro)        : {metrics['f1_macro'] * 100:.2f}%")
    print(f"Specificity (Macro)     : {metrics['specificity_macro'] * 100:.2f}%")
    print(f"Error Rate              : {metrics['error_rate'] * 100:.2f}%")
    print("-" * 60)
    
    print("="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print(f"Total Classes: {len(classes)}")
    print("="*80)
    
    print("\n[Step 7] Saving trained model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'test_accuracy': test_accuracy,
            'classes': classes,
            'num_classes': len(classes),
            'backbone': 'EfficientNet-B0 (spatial-only)'
        }, 'fruits_efficientnet_spatial_cnn.pth')
        print("Model saved as 'fruits_efficientnet_spatial_cnn.pth'")
    except Exception as e:
        print(f"ERROR saving model: {e}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\nFinal GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

if __name__ == "__main__":
    main()